# V25 POC — Mean elevation outside 2,400–3,500 m

This notebook evaluates the proposed `ELEVATION_OUT_OF_RANGE` rule against the
ignored real Kobo export in `notebooks/data`, using the same Copernicus GLO-30
subset that the V24 slope POC downloaded from OpenTopography.

Privacy rules follow V24: raw rows, names, phone numbers, attachment URLs, and
exact coordinates are never displayed, identifiers in outputs are one-way
hashes, and every generated artifact stays below the fully ignored
`notebooks/data/v25_poc_output` directory. The notebook performs no database
writes.

Unlike V24, elevation is read from the **native GLO-30 grid without
resampling**. Slope needs a metric grid because it is a gradient; elevation is
a per-pixel value, so reprojecting first would only smooth the source. Pixel
weights are the true intersection areas measured in the local UTM zone.

## 1. Environment and configuration

Run from the repository root. Required packages are `numpy`, `rasterio`,
`shapely`, `pyproj`, and `requests`. If `notebooks/data/v24_poc_output/cop30_subset.tif`
already exists, this notebook reuses it and performs no network request. This
is the shared-terrain decision (V25 D-2) exercised in practice.

In [ ]:
from pathlib import Path
import json
import math
import os
import statistics
import sys

import numpy as np
import rasterio
import requests
from shapely.geometry import box

sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from poc_common import (
    dataset_utm_crs,
    load_backend_helpers,
    load_records,
    poc_paths,
    project_polygon,
    utm_transformer,
    write_public_output,
)

PATHS = poc_paths('v25')
OUTPUT_DIR = PATHS['output_dir']
DEM_PATH = PATHS['v24_output_dir'] / 'cop30_subset.tif'
RESULT_PATH = OUTPUT_DIR / 'elevation_results_private.json'
AC_DEMO_PATH = OUTPUT_DIR / 'v25_acceptance_demo.json'

OPENTOPOGRAPHY_URL = 'https://portal.opentopography.org/API/globaldem'
DEM_SOURCE = 'Copernicus DEM GLO-30'
DEM_SOURCE_VERSION = 'DGED-2023_1'
DEM_SOURCE_PROVIDER = 'OpenTopography'
ELEVATION_MIN_M = 2400.0
ELEVATION_MAX_M = 3500.0
ELEVATION_RULE_VERSION = 'v1'
MIN_RASTER_COVERAGE_PERCENT = 90.0
DEM_CELL_AREA_M2 = 900.0
LOW_CONFIDENCE_PIXEL_EQUIVALENT = 4.0
BBOX_PADDING_DEGREES = 0.01

print({
    'csv_present': PATHS['csv'].exists(),
    'shared_v24_dem_present': DEM_PATH.exists(),
    'network_request_needed': not DEM_PATH.exists(),
})

## 2. Load the real submissions through the production parser

`poc_common.load_records` uses `utils.polygon.parse_odk_geoshape`,
`validate_polygon`, `coords_to_wkt`, and `calculate_area_ha` — the same helpers
the Kobo sync path uses — so the POC measures the polygons the application
would actually store.

In [ ]:
HELPERS = load_backend_helpers(PATHS['repo_root'])
records, valid_records = load_records(PATHS['csv'], HELPERS)

print({
    'submission_count': len(records),
    'valid_polygon_count': len(valid_records),
    'invalid_polygon_count': len(records) - len(valid_records),
    'raw_values_displayed': False,
})

## 3. Reuse or acquire the shared GLO-30 subset

Bounds are derived from the valid plots with a small padding halo and are
deliberately not printed. The OpenTopography key is read from
`OPENTOPOGRAPHY_API_KEY`, stays in process memory, and is never persisted. In
production this download is a release-time step only; neither the API nor the
Django-Q2 worker contacts OpenTopography.

In [ ]:
all_coords = [coord for record in valid_records for coord in record['coords']]
private_bbox = HELPERS['compute_bbox'](all_coords)


def is_tiff(content):
    return content[:4] in (b'II*\x00', b'MM\x00*')


def ensure_dem(destination):
    if destination.exists() and destination.stat().st_size > 0:
        return 'reused-from-v24'
    api_key = os.getenv('OPENTOPOGRAPHY_API_KEY')
    if not api_key:
        raise RuntimeError(
            'Set OPENTOPOGRAPHY_API_KEY, or run the V24 notebook first'
        )
    params = {
        'demtype': 'COP30',
        'south': private_bbox['min_lat'] - BBOX_PADDING_DEGREES,
        'north': private_bbox['max_lat'] + BBOX_PADDING_DEGREES,
        'west': private_bbox['min_lon'] - BBOX_PADDING_DEGREES,
        'east': private_bbox['max_lon'] + BBOX_PADDING_DEGREES,
        'outputFormat': 'GTiff',
        'API_Key': api_key,
    }
    response = requests.get(OPENTOPOGRAPHY_URL, params=params, timeout=300)
    response.raise_for_status()
    if not is_tiff(response.content):
        raise RuntimeError('OpenTopography did not return a GeoTIFF')
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(response.content)
    return 'downloaded'


dem_state = ensure_dem(DEM_PATH)
with rasterio.open(DEM_PATH) as dem:
    DEM_CRS = dem.crs
    DEM_NODATA = dem.nodata
    print({
        'dem_state': dem_state,
        'crs': str(DEM_CRS),
        'shape': dem.shape,
        'nodata': DEM_NODATA,
        'dtype': dem.dtypes[0],
        'bounds_printed': False,
    })

## 4. Area-weighted mean elevation per plot

For every plot the notebook reads only the intersecting DEM window, builds the
exact intersection polygon between the plot and each source cell, and weights
each elevation by that intersection area measured in the local UTM zone.

`coverage_percent` is the share of the plot covered by valid (non-nodata)
pixels. A result below `MIN_RASTER_COVERAGE_PERCENT` is recorded as
`unavailable` rather than as a pass. `effective_pixel_equivalent` is the plot
area divided by the 900 m² GLO-30 cell area and carries the same
resolution-confidence caveat V24 found.

In [ ]:
def elevation_flag(mean_elevation_m):
    below = mean_elevation_m < ELEVATION_MIN_M
    above = mean_elevation_m > ELEVATION_MAX_M
    if not (below or above):
        return None
    direction = 'below' if below else 'above'
    return {
        'type': 'ELEVATION_OUT_OF_RANGE',
        'severity': 'warning',
        'note': (
            f'Mean elevation is {mean_elevation_m:,.2f} m, which is '
            f'{direction} the expected range of '
            f'{ELEVATION_MIN_M:,.0f}-{ELEVATION_MAX_M:,.0f} m. '
            f'Dataset source: {DEM_SOURCE} {DEM_SOURCE_VERSION} via '
            f'{DEM_SOURCE_PROVIDER}.'
        ),
    }


UTM_CRS = dataset_utm_crs(valid_records)
TO_UTM = utm_transformer(UTM_CRS)


def intersecting_cells(dem, polygon_wgs84, pad=1):
    """Yield (row, col, cell polygon) for every source cell touching a plot."""
    min_x, min_y, max_x, max_y = polygon_wgs84.bounds
    top_row, left_col = dem.index(min_x, max_y, op=math.floor)
    bottom_row, right_col = dem.index(max_x, min_y, op=math.ceil)
    top_row = max(top_row - pad, 0)
    left_col = max(left_col - pad, 0)
    bottom_row = min(bottom_row + pad, dem.height - 1)
    right_col = min(right_col + pad, dem.width - 1)
    for row in range(top_row, bottom_row + 1):
        for col in range(left_col, right_col + 1):
            x0, y0 = dem.transform * (col, row)
            x1, y1 = dem.transform * (col + 1, row + 1)
            cell = box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))
            if cell.intersects(polygon_wgs84):
                yield row, col, cell


def measure_elevation(polygon_wgs84, dem):
    """Return the area-weighted mean elevation and its quality details."""
    plot_utm = project_polygon(polygon_wgs84, TO_UTM)
    plot_area_m2 = plot_utm.area
    values = []
    weights = []
    covered_area_m2 = 0.0
    for row, col, cell in intersecting_cells(dem, polygon_wgs84):
        piece = project_polygon(cell, TO_UTM).intersection(plot_utm)
        if piece.is_empty or piece.area <= 0:
            continue
        value = float(dem.read(1, window=((row, row + 1), (col, col + 1)))[0][0])
        nodata = dem.nodata
        if nodata is not None and value == nodata:
            continue
        values.append(value)
        weights.append(piece.area)
        covered_area_m2 += piece.area

    coverage_percent = (
        covered_area_m2 / plot_area_m2 * 100.0 if plot_area_m2 else 0.0
    )
    effective_pixels = plot_area_m2 / DEM_CELL_AREA_M2
    details = {
        'valid_pixel_count': len(values),
        'coverage_percent': round(coverage_percent, 3),
        'plot_area_m2': round(plot_area_m2, 2),
        'effective_pixel_equivalent': round(effective_pixels, 3),
        'pixel_resolution_m': 30,
        'resolution_confidence': (
            'low'
            if effective_pixels < LOW_CONFIDENCE_PIXEL_EQUIVALENT
            else 'normal'
        ),
    }
    if not values or coverage_percent < MIN_RASTER_COVERAGE_PERCENT:
        return None, details
    return float(np.average(values, weights=weights)), details


results = []
with rasterio.open(DEM_PATH) as dem:
    for record in valid_records:
        mean_elevation, details = measure_elevation(record['polygon'], dem)
        if mean_elevation is None:
            results.append({
                'plot_ref': record['plot_ref'],
                'status': 'unavailable',
                'value': None,
                'details': details,
                'flag': None,
            })
            continue
        flag = elevation_flag(mean_elevation)
        results.append({
            'plot_ref': record['plot_ref'],
            'status': 'complete',
            'value': round(mean_elevation, 3),
            'details': details,
            'flag': flag,
        })

complete_results = [r for r in results if r['status'] == 'complete']
print({
    'analysed_plots': len(results),
    'complete': len(complete_results),
    'unavailable': len(results) - len(complete_results),
})

## 5. Aggregate outcome of the proposed rule

The table below is the evidence product and GIS need in order to accept or
change the 2,400–3,500 m range.

In [ ]:
values = [r['value'] for r in complete_results]
flagged = [r for r in complete_results if r['flag']]
below = [r for r in flagged if r['value'] < ELEVATION_MIN_M]
above = [r for r in flagged if r['value'] > ELEVATION_MAX_M]
low_confidence = [
    r for r in complete_results
    if r['details']['resolution_confidence'] == 'low'
]

summary = {
    'valid_input_polygons': len(valid_records),
    'completed_calculations': len(complete_results),
    'flagged_out_of_range': len(flagged),
    'flagged_below_2400': len(below),
    'flagged_above_3500': len(above),
    'min_mean_elevation_m': round(min(values), 2),
    'median_mean_elevation_m': round(statistics.median(values), 2),
    'max_mean_elevation_m': round(max(values), 2),
    'min_coverage_percent': round(
        min(r['details']['coverage_percent'] for r in complete_results), 2
    ),
    'min_intersecting_cells': min(
        r['details']['valid_pixel_count'] for r in complete_results
    ),
    'max_intersecting_cells': max(
        r['details']['valid_pixel_count'] for r in complete_results
    ),
    'low_resolution_confidence_plots': len(low_confidence),
}
print(json.dumps(summary, indent=2))

## 6. Boundary-contract demo using existing records

The observed data may not contain a naturally out-of-range plot, so the
strict-boundary contract is demonstrated deterministically. Each scenario
reuses an existing hashed plot record and its real measured value, but
substitutes a controlled evaluation value. Controlled values are labelled and
never replace the observed terrain measurement.

In [ ]:
if len(complete_results) < 4:
    raise RuntimeError('At least four complete results are required')

controlled_scenarios = [
    ('below_range', complete_results[0], 2399.99),
    ('exactly_lower_bound', complete_results[1], 2400.00),
    ('exactly_upper_bound', complete_results[2], 3500.00),
    ('above_range', complete_results[3], 3500.01),
]

acceptance_rows = []
for scenario, real_result, controlled_value in controlled_scenarios:
    flag = elevation_flag(controlled_value)
    acceptance_rows.append({
        'scenario': scenario,
        'uses_existing_hashed_plot': True,
        'plot_ref': real_result['plot_ref'],
        'observed_mean_elevation_m': real_result['value'],
        'controlled_demo_mean_elevation_m': controlled_value,
        'controlled_demo_input': True,
        'flagged_for_review': bool(flag),
        'flagged_reason': [flag] if flag else [],
        'geospatial_metrics': {
            'elevation': {
                'status': 'complete',
                'value': controlled_value,
                'unit': 'metres',
                'source': DEM_SOURCE,
                'source_version': DEM_SOURCE_VERSION,
                'source_provider': DEM_SOURCE_PROVIDER,
                'threshold': {
                    'operator': 'outside',
                    'min': ELEVATION_MIN_M,
                    'max': ELEVATION_MAX_M,
                    'unit': 'metres',
                    'rule_version': ELEVATION_RULE_VERSION,
                },
                'details': real_result['details'],
            }
        },
    })

AC_DEMO_PATH.write_text(json.dumps(acceptance_rows, indent=2))
print([
    (row['scenario'], row['flagged_for_review'])
    for row in acceptance_rows
])

## 7. Persist the POC artifacts

Output is split by sensitivity.

`notebooks/outputs/` is **tracked by git**, so it receives only the aggregate
summary, the dataset provenance, and the boundary demo — no per-plot rows. This
is the evidence the plan cites, and a reviewer can read it in a pull request.

`notebooks/data/v25_poc_output/` stays **fully ignored** and keeps the per-plot
results. Elevation rows carry no coordinates, but 166 of them alongside plot
areas are still per-subject records, so they are not committed by default.

In [ ]:
def metric_document(result):
    return {
        'status': result['status'],
        'value': result['value'],
        'unit': 'metres',
        'source': DEM_SOURCE,
        'source_version': DEM_SOURCE_VERSION,
        'source_provider': DEM_SOURCE_PROVIDER,
        'threshold': {
            'operator': 'outside',
            'min': ELEVATION_MIN_M,
            'max': ELEVATION_MAX_M,
            'unit': 'metres',
            'rule_version': ELEVATION_RULE_VERSION,
        },
        'details': result['details'],
    }


payload = {
    'summary': summary,
    'dataset': {
        'source': DEM_SOURCE,
        'source_version': DEM_SOURCE_VERSION,
        'source_provider': DEM_SOURCE_PROVIDER,
        'reused_v24_subset': dem_state == 'reused-from-v24',
    },
    'results': [
        {
            'plot_ref': result['plot_ref'],
            'geospatial_metrics': {'elevation': metric_document(result)},
            'flagged_reason': [result['flag']] if result['flag'] else [],
        }
        for result in results
    ],
}
RESULT_PATH.write_text(json.dumps(payload, indent=2))

public = write_public_output('v25_elevation_summary.json', {
    'poc': 'v25',
    'metric': 'elevation',
    'notebook': 'notebooks/v25_elevation_range_poc.ipynb',
    'dataset': payload['dataset'],
    'summary': summary,
    'boundary_demo': [
        {
            'scenario': row['scenario'],
            'controlled_demo_mean_elevation_m': (
                row['controlled_demo_mean_elevation_m']
            ),
            'flagged_for_review': row['flagged_for_review'],
            'note': (
                row['flagged_reason'][0]['note']
                if row['flagged_reason'] else None
            ),
        }
        for row in acceptance_rows
    ],
})

print({
    'ignored_private_files': [RESULT_PATH.name, AC_DEMO_PATH.name],
    'tracked_public_file': public['file'],
})

## 8. Synthetic example — what a flagged plot looks like

The real data never trips this rule — every observed plot sits between 2,569 m and 2,921 m, so the positive branch has no
visual evidence behind it. This section hand-draws a 25 m example plot at a
**public landmark far from the collection area**, measures it with the same
function used above, and previews it on a basemap.

Two properties make this safe to commit: the polygon is fabricated, and the
location is a named landmark rather than a farm, so the map discloses nothing
about where African Bamboo works. Elevation comes from the keyless Copernicus GLO-30 mirror on AWS Open Data, the same product the production asset package is cut from, so no API key is needed.

In [ ]:
from poc_common import (
    DEMO_LOCATIONS,
    preview_map,
    public_dem_url,
    synthetic_square,
)

DEMO_SIDE_M = 25.0
demo_lon, demo_lat, demo_place = DEMO_LOCATIONS['high_elevation']
demo_plot = synthetic_square(demo_lon, demo_lat, DEMO_SIDE_M)

# UTM zone 37N covers 36-42E, so the dataset transformer built above is valid
# at this demo location too.
with rasterio.open(public_dem_url(demo_lon, demo_lat)) as demo_dem:
    demo_value, demo_details = measure_elevation(demo_plot, demo_dem)

demo_flag = elevation_flag(demo_value) if demo_value is not None else None
print(json.dumps({
    'location': demo_place,
    'mean_elevation_m': round(demo_value, 2) if demo_value else None,
    'expected_range_m': [ELEVATION_MIN_M, ELEVATION_MAX_M],
    'flagged': bool(demo_flag),
    'note': demo_flag['note'] if demo_flag else None,
    'coverage_percent': demo_details['coverage_percent'],
    'valid_pixel_count': demo_details['valid_pixel_count'],
}, indent=2))

The map below is the visual check. A red outline means the rule
fired; click the polygon for the measured values. Run
`pip install -r notebooks/requirements.txt` if folium is missing.

In [ ]:
preview_map(
    demo_plot,
    flagged=bool(demo_flag),
    title=f'V25 example - {demo_place}',
    rows={
        'Mean elevation': f'{demo_value:,.1f} m',
        'Expected range': (
            f'{ELEVATION_MIN_M:,.0f}-{ELEVATION_MAX_M:,.0f} m'
        ),
        'Result': 'ELEVATION_OUT_OF_RANGE' if demo_flag else 'pass',
        'Source': f'{DEM_SOURCE} {DEM_SOURCE_VERSION}',
    },
    zoom=16,
)

## 8. Review checklist and interpretation

The POC is technically successful when every valid geoshape parses, the shared
GLO-30 subset covers all plots, means are deterministic, insufficient coverage
is reported as `unavailable` rather than as a pass, and the strict-boundary
demo shows that exactly `2,400.00` and exactly `3,500.00` are not flagged.

Two product questions remain and are answered by the printed summary rather
than by this notebook:

1. Does the observed elevation distribution justify the 2,400–3,500 m range,
   or does the collection area sit predominantly outside it?
2. Is GLO-30 acceptable when a plot occupies a fraction of one 30 m cell? The
   `low_resolution_confidence_plots` count carries the same caveat V24 raised;
   for elevation the effect is smaller than for slope, because elevation varies
   far less than gradient across a single cell.

For an independent QGIS spot check, load the ignored
`notebooks/data/v24_poc_output/cop30_subset.tif` and compare several hashed
records with Zonal Statistics. Do not copy the real polygons or the raster
outside the ignored POC directory.

Citation: European Space Agency (2024), Copernicus Global Digital Elevation
Model, distributed by OpenTopography.